# spVIPES on PBMC CITE-seq — Vaccination Time-Course

This vignette demonstrates **spVIPES** on a real-world CITE-seq dataset
from [Hao *et al.* (2021)](https://doi.org/10.1016/j.cell.2021.04.048),
available via [`scvi.data.pbmc_seurat_v4_cite_seq()`](https://docs.scvi-tools.org/en/1.4.1/api/reference/scvi.data.pbmc_seurat_v4_cite_seq.html).

**Dataset overview.** 161,764 peripheral blood mononuclear cells (PBMCs)
from 8 volunteers enrolled in an HIV vaccine trial, measured with CITE-seq
(simultaneous RNA + surface protein) at **three time points**: day 0
(baseline), day 3, and day 7 post-vaccination.

**What we showcase:**

| Feature | API |
|---|---|
| **3-group integration** (day 0 / day 3 / day 7) | `spVIPES.data.prepare_adatas`, label-based PoE |
| **Shared latent** captures cell-type identity across time points | `get_latent_representation` → shared UMAP |
| **Private latents** capture time-point-specific variation | per-group private UMAP |
| **Training diagnostics** | ELBO training curve |
| **Gene loadings** | `get_loadings()` — top genes per latent factor |

> **Runtime:** The notebook subsamples to 2,000 cells per time point and
> 2,000 highly variable genes so it runs on a laptop CPU in ~10 minutes.
> For a full run, increase `N_PER_GROUP` and `N_HVG` and train on GPU.

## 1. Environment

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import scipy.sparse as sp

import spVIPES

np.random.seed(42)
torch.manual_seed(42)
sc.settings.set_figure_params(dpi=80, frameon=False)

print(f"spVIPES   : {spVIPES.__version__}")
print(f"scvi-tools: {scvi.__version__}")
print(f"scanpy    : {sc.__version__}")
print(f"torch     : {torch.__version__} (CUDA: {torch.cuda.is_available()})")

## 2. Load the PBMC CITE-seq dataset

[`scvi.data.pbmc_seurat_v4_cite_seq()`](https://docs.scvi-tools.org/en/1.4.1/api/reference/scvi.data.pbmc_seurat_v4_cite_seq.html)
downloads the Hao *et al.* (2021) CITE-seq dataset. It contains:

- **RNA** counts (raw UMI) in `adata.layers["counts"]` and log-normalised values in `adata.X`,
- **Protein** abundances (228 TotalSeq-A antibodies) in `adata.obsm["protein_counts"]`,
- Cell-type annotations, donor IDs, and time-point metadata in `adata.obs`.

The 8 donors were each sampled at **day 0, day 3, and day 7** following
vaccination — giving us a natural 3-group split for spVIPES.

In [ ]:
adata_full = scvi.data.pbmc_seurat_v4_cite_seq(save_path="./data/")
adata_full.obs_names_make_unique()

print("Shape         :", adata_full.shape)
print("Obs columns   :", list(adata_full.obs.columns[:15]))
print()
print(adata_full.obs.head(3))

## 3. Identify time points and assign groups

We extract the vaccination time point from the `orig.ident` (or batch)
column. Each donor–time combination forms a batch; the three time points
(day 0, day 3, day 7) define our **three groups** for spVIPES.

This is a biologically motivated split: we expect shared cell-type
identity across all time points (captured by the **shared latent**),
while vaccination-induced changes should appear as time-point-specific
variation (captured by the **private latents**).

In [ ]:
# Inspect available batch / donor information
print("Unique values in 'orig.ident':")
print(adata_full.obs["orig.ident"].value_counts())

In [ ]:
# Extract the time point from the batch identifier
# The orig.ident column encodes donor and time, e.g. "P1_0", "P1_3", "P1_7"
# We parse the day from the last part after the underscore
def extract_time_point(ident):
    """Extract the day (0, 3, or 7) from the orig.ident string."""
    day = ident.split("_")[-1]
    return f"day_{day}"

adata_full.obs["time_point"] = adata_full.obs["orig.ident"].apply(extract_time_point)
adata_full.obs["time_point"] = adata_full.obs["time_point"].astype("category")

print("Cells per time point:")
print(adata_full.obs["time_point"].value_counts().sort_index())
print()

# Also check what cell-type annotations are available
if "celltype.l1" in adata_full.obs.columns:
    label_col = "celltype.l1"
elif "celltype.l2" in adata_full.obs.columns:
    label_col = "celltype.l2"
else:
    # Fall back: find the first column that looks like a cell-type annotation
    label_col = [c for c in adata_full.obs.columns if "cell" in c.lower() and "type" in c.lower()]
    label_col = label_col[0] if label_col else None

print(f"Cell-type label column: '{label_col}'")
if label_col:
    print(f"Number of cell types : {adata_full.obs[label_col].nunique()}")
    print(adata_full.obs[label_col].value_counts().head(10))

## 4. Subsample and select highly variable genes

To keep runtime tractable on CPU we:

- subsample **2,000 cells per time point** (6,000 total),
- select the **top 2,000 highly variable genes**,
- and use `adata.layers["counts"]` (raw UMI counts) as input to spVIPES.

For a full-scale run on GPU, increase `N_PER_GROUP` and `N_HVG`.

In [ ]:
N_PER_GROUP = 2000   # cells per time point
N_HVG = 2000         # highly variable genes

# --- Subsample cells ---
rng = np.random.default_rng(42)
time_col = adata_full.obs["time_point"].values
pos_keep = []
for tp in sorted(adata_full.obs["time_point"].unique()):
    pos = np.where(time_col == tp)[0]
    pick = rng.choice(pos, size=min(N_PER_GROUP, len(pos)), replace=False)
    pos_keep.extend(pick)
pos_keep = np.array(sorted(pos_keep))
adata = adata_full[pos_keep].copy()
adata.obs_names_make_unique()

# --- Use raw counts as .X for spVIPES ---
if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()
    print("Using raw counts from adata.layers['counts']")

# --- Select top-N_HVG genes by variance ---
X = adata.X
if sp.issparse(X):
    gene_var = np.asarray(X.power(2).mean(axis=0)).flatten() - np.asarray(X.mean(axis=0)).flatten() ** 2
else:
    gene_var = np.var(X, axis=0)
top_gene_idx = np.argsort(gene_var)[::-1][:N_HVG]
adata = adata[:, top_gene_idx].copy()

print(f"After subsample + HVG : {adata.shape}")
print(f"Cells per time point  :")
print(adata.obs["time_point"].value_counts().sort_index())

## 5. Split into three groups and prepare for spVIPES

We split the subsampled AnnData into three separate AnnData objects — one
per time point — and feed them into `spVIPES.data.prepare_adatas()`.

This function concatenates the groups, adds group-level bookkeeping to
`adata.uns`, and appends group names to feature names (required for
spVIPES' internal indexing).

In [ ]:
# Split into per-time-point AnnData objects
time_points = sorted(adata.obs["time_point"].unique())
adatas_dict = {}
for tp in time_points:
    sub = adata[adata.obs["time_point"] == tp].copy()
    # Clean up — prepare_adatas doesn't need extra obsm/uns
    sub.uns = {}
    sub.obsm = {}
    sub.layers = {}
    adatas_dict[tp] = sub
    print(f"  {tp}: {sub.shape}")

# Concatenate with spVIPES
adata_spv = spVIPES.data.prepare_adatas(adatas_dict)

print(f"\nConcatenated AnnData: {adata_spv.shape}")
print(f"Groups             : {list(adata_spv.uns['groups_mapping'].values())}")
print(f"Group sizes        : {[len(g) for g in adata_spv.uns['groups_obs_indices']]}")

## 6. Set up the model — label-based PoE

With **three groups**, we use the **label-based Product of Experts (PoE)**
strategy. This requires a cell-type label column in `adata.obs`.
`setup_anndata` will match cells across groups that share the same label
and combine their encoder outputs through the PoE to produce the shared
latent representation.

Cell types present in one time point but absent in another are handled
gracefully — the missing group contributes an uninformative prior N(0, I).

In [ ]:
spVIPES.model.spVIPES.setup_anndata(
    adata_spv,
    groups_key="groups",
    label_key=label_col,
)

## 7. Instantiate and train the model

We configure the model with:

- **`n_dimensions_shared=15`**: the shared latent captures cell-type identity common across all three time points.
- **`n_dimensions_private=8`**: each group gets its own private latent to capture time-point-specific variation.
- **`n_hidden=128`**: width of the hidden layer in encoders and decoders.
- Training uses KL warmup over the first 30 epochs to stabilise early training.

In [ ]:
# Model hyperparameters
N_SHARED   = 15
N_PRIVATE  = 8
N_HIDDEN   = 128
DROPOUT    = 0.1
MAX_EPOCHS = 50
BATCH_SIZE = 128
KL_WARMUP  = 30

model = spVIPES.model.spVIPES(
    adata_spv,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
)
print(model)

In [ ]:
# Get group indices for training
group_indices_list = [
    np.where(adata_spv.obs["groups"] == group)[0]
    for group in adata_spv.obs["groups"].unique()
]
print("Group sizes:", [len(g) for g in group_indices_list])

# Train
model.train(
    group_indices_list,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    train_size=0.9,
    early_stopping=False,
    n_epochs_kl_warmup=KL_WARMUP,
)

## 8. Training diagnostics

We plot the ELBO training loss to verify that the model converged.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(model.history["elbo_train"]["elbo_train"], label="ELBO (train)")
if "elbo_validation" in model.history:
    ax.plot(model.history["elbo_validation"]["elbo_validation"], label="ELBO (val)")
ax.set_xlabel("Epoch")
ax.set_ylabel("ELBO")
ax.set_title("spVIPES training curve")
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

## 9. Extract latent representations

spVIPES returns three sets of latent variables:

- **Shared latent** (one per cell): captures variation common to all time points — primarily cell-type identity.
- **Private latents** (one per group): capture time-point-specific variation — vaccination-induced changes.

The `_reordered` variants are aligned back to the original cell order in `adata_spv`.

In [ ]:
latents = model.get_latent_representation(group_indices_list, batch_size=BATCH_SIZE)

print("Latent keys:", list(latents.keys()))
for g, tp in enumerate(time_points):
    print(f"  {tp}  shared: {latents['shared_reordered'][g].shape}  "
          f"private: {latents['private_reordered'][g].shape}")

In [ ]:
# Stitch the shared latent back into the full AnnData in original cell order
latent_shared = np.concatenate(
    [latents["shared_reordered"][g] for g in range(len(time_points))],
    axis=0,
)

# Trim adata_spv to match latent size (in case of train_size < 1.0 rounding)
adata_spv = adata_spv[:len(latent_shared)].copy()
adata_spv.obsm["X_spVIPES_shared"] = latent_shared

## 10. Visualise the shared latent space

The shared latent should integrate cells across time points while
preserving cell-type identity. We compute UMAP on the shared embedding
and colour by:

1. **Cell type** — clusters should reflect biological identity.
2. **Time point** — groups should be well-mixed within each cluster, indicating successful integration.

In [ ]:
# Compute UMAP on shared embedding
sc.pp.neighbors(adata_spv, use_rep="X_spVIPES_shared", key_added="spvipes_shared", n_neighbors=15)
sc.tl.umap(adata_spv, neighbors_key="spvipes_shared", min_dist=0.3)
adata_spv.obsm["X_umap_shared"] = adata_spv.obsm["X_umap"].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc.pl.embedding(
    adata_spv, basis="X_umap_shared", color=label_col,
    ax=axes[0], show=False, title="Shared latent — cell types",
    legend_loc="on data", legend_fontsize=6, size=8,
)
sc.pl.embedding(
    adata_spv, basis="X_umap_shared", color="time_point",
    ax=axes[1], show=False, title="Shared latent — time points",
    palette="Set2", size=8,
)

sns.despine()
plt.tight_layout()
plt.show()

## 11. Visualise the private latent spaces

Each time point has its own private latent space. The private latent should
capture variation **specific** to that time point — e.g., vaccination-induced
transcriptional changes at day 3 or day 7 that are not present at baseline.

Ideally, private spaces should:
- Show internal structure reflecting time-point-specific biology.
- **Not** separate cells by cell type (that information belongs in the shared space).

In [ ]:
# Build per-group sub-AnnDatas with private embeddings
group_adatas = {}
for g, tp in enumerate(time_points):
    mask = adata_spv.obs["groups"] == tp
    sub = adata_spv[mask].copy()
    sub.obsm["X_spVIPES_private"] = latents["private_reordered"][g]
    sc.pp.neighbors(sub, use_rep="X_spVIPES_private", key_added="spvipes_private", n_neighbors=15)
    sc.tl.umap(sub, neighbors_key="spvipes_private", min_dist=0.3)
    sub.obsm["X_umap_private"] = sub.obsm["X_umap"].copy()
    group_adatas[tp] = sub

# Plot private latents coloured by cell type (should show little structure)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, tp in zip(axes, time_points):
    sc.pl.embedding(
        group_adatas[tp], basis="X_umap_private", color=label_col,
        ax=ax, show=False, title=f"Private latent — {tp}",
        legend_loc=None, size=10,
    )
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Plot private latents coloured by donor to see if private space captures donor effects
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, tp in zip(axes, time_points):
    sc.pl.embedding(
        group_adatas[tp], basis="X_umap_private", color="orig.ident",
        ax=ax, show=False, title=f"Private latent — {tp} (donors)",
        size=10,
    )
sns.despine()
plt.tight_layout()
plt.show()

## 12. Gene loadings — which genes drive each latent factor?

spVIPES uses a linear decoder, so we can extract per-gene weights (loadings)
linking each latent dimension to gene expression. This is useful for
interpreting what biological signals each latent factor captures.

We inspect the top genes per shared and private latent dimension.

In [ ]:
loadings = model.get_loadings()

print("Loading keys (group, latent_type):")
for key in loadings:
    print(f"  {key}: {loadings[key].shape}")

In [ ]:
# Top 5 genes per shared latent dimension (from the first group's decoder)
first_group_key = [k for k in loadings if "shared" in k[1]][0]
shared_loadings = loadings[first_group_key]

print(f"Top 5 genes per shared dimension ({first_group_key[0]} decoder):\n")
top_genes = shared_loadings.apply(lambda col: col.abs().nlargest(5).index.tolist())
print(top_genes.to_string())

In [ ]:
# Heatmap of absolute shared loadings (top 10 genes per dimension)
n_top = 10
top_genes_set = set()
for col in shared_loadings.columns:
    top_genes_set.update(shared_loadings[col].abs().nlargest(n_top).index)

loadings_subset = shared_loadings.loc[list(top_genes_set)]

fig, ax = plt.subplots(figsize=(12, max(6, len(top_genes_set) * 0.25)))
sns.heatmap(
    loadings_subset.values,
    xticklabels=loadings_subset.columns,
    yticklabels=loadings_subset.index,
    cmap="RdBu_r", center=0, ax=ax,
    cbar_kws={"label": "Loading weight"},
)
ax.set_title(f"Shared decoder loadings — {first_group_key[0]}")
ax.set_xlabel("Shared latent dimension")
ax.set_ylabel("Gene")
plt.tight_layout()
plt.show()

## 13. Quantitative evaluation

We evaluate the shared latent space with two complementary metrics:

- **Biological conservation (ARI):** Adjusted Rand Index between Leiden
  clusters on the shared embedding and ground-truth cell-type labels.
  Higher is better — the embedding preserves cell identity.
- **Batch mixing (entropy):** Normalised entropy of the time-point
  distribution inside k-nearest-neighbour balls. Higher is better (1.0 =
  perfect mixing across time points).

A successful shared latent achieves **high ARI and high batch mixing
simultaneously**.

In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

def leiden_ari(rep, labels, resolution=0.8):
    """Compute ARI between Leiden clusters on `rep` and ground-truth `labels`."""
    tmp = ad.AnnData(X=rep)
    sc.pp.neighbors(tmp, use_rep="X", n_neighbors=15)
    sc.tl.leiden(tmp, resolution=resolution, random_state=0)
    return adjusted_rand_score(labels, tmp.obs["leiden"].values)

def batch_entropy(rep, groups_obs, k=30):
    """Mean normalised entropy of group composition in k-NN balls."""
    groups_obs = np.asarray(groups_obs)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(rep)
    _, idx = nn.kneighbors(rep)
    idx = idx[:, 1:]  # drop self
    n_groups = len(np.unique(groups_obs))
    max_ent = np.log2(n_groups)
    entropies = []
    for i in range(rep.shape[0]):
        _, counts = np.unique(groups_obs[idx[i]], return_counts=True)
        p = counts / counts.sum()
        H = -np.sum(p * np.log2(p + 1e-12))
        entropies.append(H / max_ent)
    return float(np.mean(entropies))

rep = adata_spv.obsm["X_spVIPES_shared"]
ari = leiden_ari(rep, adata_spv.obs[label_col].values)
mix = batch_entropy(rep, adata_spv.obs["time_point"].values, k=30)

results_df = pd.DataFrame([{
    "ARI (cell types)": round(ari, 3),
    "Batch mixing (time points)": round(mix, 3),
}], index=["spVIPES shared"])

print(results_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
x = np.arange(len(results_df.columns))
width = 0.5
bars = ax.bar(x, results_df.values[0], width, color=["#4C72B0", "#55A868"])
ax.set_xticks(x)
ax.set_xticklabels(results_df.columns, fontsize=10)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("Shared latent quality")
for bar, val in zip(bars, results_df.values[0]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", fontsize=11, fontweight="bold")
sns.despine()
plt.tight_layout()
plt.show()

## 14. Summary

In this vignette we demonstrated spVIPES on the Hao *et al.* (2021)
PBMC CITE-seq vaccination time-course dataset:

1. **Three-group integration** — cells from day 0, day 3, and day 7
   were integrated using the label-based Product of Experts.
2. **Shared latent space** — captures cell-type identity while mixing
   cells from all three time points, as confirmed by both UMAP
   visualisation and quantitative ARI / batch-mixing scores.
3. **Private latent spaces** — one per time point, designed to capture
   time-point-specific variation (vaccination response) without
   recapitulating shared cell-type structure.
4. **Gene loadings** — the linear decoder weights reveal which genes
   drive each latent dimension, aiding biological interpretation.

### Next steps

- **Multimodal integration:** This dataset contains protein measurements
  alongside RNA. See the [multimodal tutorial](multimodal_nf_tutorial.ipynb)
  for how to use `prepare_multimodal_adatas` with both RNA and protein
  modalities.
- **Normalizing flow priors:** Replace the standard Gaussian prior with
  Neural Spline Flows (`use_nf_prior=True`) for more expressive latent
  distributions.
- **Full-scale run:** Increase `N_PER_GROUP` and `N_HVG`, train for more
  epochs on GPU, and use `early_stopping=True` for optimal performance.

### References

- Hao, Y. *et al.* (2021). *Integrated analysis of multimodal single-cell
  data.* Cell 184, 3573-3587.
- Novella-Rausell, C., Peters, D.J.M., & Mahfouz, A. (2023).
  *Integrative learning of disentangled representations in multi-view
  single-cell data.*